# gpudb on a free GPU, in a few minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/singhpratech/duckdbgpumetaldbram/blob/main/examples/gpudb_quickstart.ipynb)

**gpudb** is a DuckDB community extension that runs aggregates, joins and `GROUP BY` /
`HAVING` / top-k on the GPU — NVIDIA CUDA on Linux, Apple Silicon Metal on macOS.

What this notebook does, all against native DuckDB **in the same process, results
checked equal in the same cell**:

1. get the CUDA backend onto Colab's free T4 (the community registry's Linux binary is
   CPU-only, so this loads the release binary — or builds one with Colab's `nvcc`);
2. generate TPC-H data with DuckDB's built-in `tpch` extension;
3. `GROUP BY` + `HAVING` (TPC-H Q18's inner query) and top-10 groups — v0.6.0;
4. a fused join + aggregate — v0.5.0;
5. a resident-column `SUM` — v0.4.0;
6. the honest rows: where native still wins.

This notebook asks Colab for a **T4 GPU runtime** automatically. If the GPU check below
says there is no NVIDIA GPU, use **Runtime → Change runtime type → T4 GPU** and re-run —
everything still runs on the CPU fallback, but then you are timing the fallback.


## 1. Get the CUDA backend onto this runtime

In [ ]:
%pip install -q --upgrade duckdb
import duckdb, time, os, shutil, subprocess, urllib.request
print("DuckDB", duckdb.__version__)
has_gpu = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0
if has_gpu:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())
else:
    print("No NVIDIA GPU on this runtime: Runtime -> Change runtime type -> T4 GPU, then re-run.")


In [ ]:
# The community registry's Linux binary is CPU-only (its build machines have no CUDA
# toolchain). The release binary has the CUDA backend with a statically linked runtime
# and needs only the driver; if this runtime's driver is older than it requires, we
# build the extension here with Colab's own nvcc instead (a few minutes, sm_75 only).
RELEASE = "https://github.com/singhpratech/duckdbgpumetaldbram/releases/latest/download/gpudb.linux_amd64.duckdb_extension"
ext = "/content/gpudb.linux_amd64.duckdb_extension"

def connect_with(path):
    c = duckdb.connect(config={'allow_unsigned_extensions': 'true'})
    c.sql(f"LOAD '{path}';")
    return c, c.sql("SELECT gpu_build_info()").fetchone()[0]

con = None
if has_gpu:
    urllib.request.urlretrieve(RELEASE, ext)
    try:
        con, info = connect_with(ext)
        print("release binary:", info)
        if 'runtime=cuda' not in info:
            print("release binary loaded but the CUDA runtime did not initialise on this driver — building from source instead")
            con = None
    except Exception as e:
        print("release binary could not load here:", str(e).splitlines()[0])
        con = None
    if con is None:
        os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
        assert shutil.which('nvcc'), "no nvcc on this runtime"
        print("building gpudb with this runtime's nvcc (sm_75 only) — a few minutes on Colab's 2 cores ...")
        r = subprocess.run("rm -rf /content/duckdbgpumetaldbram && cd /content && "
                           "git clone -q --depth 1 https://github.com/singhpratech/duckdbgpumetaldbram.git && cd duckdbgpumetaldbram && "
                           "CUDAARCHS=75 GPUDB_REQUIRE_CUDA=1 ./scripts/build.sh > build.log 2>&1; echo rc=$?; "
                           "grep -E 'CUDA enabled|packaging' build.log; tail -n 5 build.log",
                           shell=True, capture_output=True, text=True)
        print(r.stdout, r.stderr)
        built = "/content/duckdbgpumetaldbram/build-linux/src/extension/gpudb.linux_amd64.duckdb_extension"
        assert os.path.exists(built), "build failed — see /content/duckdbgpumetaldbram/build.log"
        con, info = connect_with(built)
        print("source build:", info)
else:
    con = duckdb.connect()
    con.sql("INSTALL gpudb FROM community; LOAD gpudb;")
    info = con.sql("SELECT gpu_build_info()").fetchone()[0]
    print("community binary (CPU fallback):", info)

GPU = 'runtime=cuda' in info
print("GPU backend live:", GPU)


## 2. TPC-H data

DuckDB's `tpch` extension generates the data in-process. SF3 (18M `lineitem` rows,
4.5M orders) fits Colab's memory comfortably; raise `sf` if you have a bigger box.


In [ ]:
con.sql("INSTALL tpch; LOAD tpch;")
t0 = time.perf_counter()
con.sql("CALL dbgen(sf = 3);")
print(f"dbgen: {time.perf_counter()-t0:.1f} s")
print(con.sql("SELECT count(*) AS lineitem_rows, count(DISTINCT l_orderkey) AS orders FROM lineitem").fetchall())

def bench(sql, runs=3):
    """Best-of-runs wall time of a full statement, first run discarded (warm-up)."""
    con.sql(sql).fetchall()
    best = float('inf')
    for _ in range(runs):
        t = time.perf_counter(); con.sql(sql).fetchall(); best = min(best, time.perf_counter() - t)
    return best * 1000

def show(label, native_ms, gpu_ms, extra=""):
    print(f"{label:52s} native {native_ms:8.1f} ms   gpudb {gpu_ms:8.1f} ms   {native_ms/gpu_ms:5.1f}x  {extra}")


## 3. GROUP BY / HAVING / top-k on the GPU (v0.6.0)

Upload the `(l_orderkey, l_quantity)` pair once. The GPU sorts it once and caches the
order; every `GROUP BY` after that is a segmented reduce, and the `_having` / `_topk`
forms apply the `HAVING` or the `ORDER BY sum LIMIT k` **on the device** so only the
surviving rows come back.


In [ ]:
t0 = time.perf_counter()
n = con.sql("SELECT gpu_upload_pair('l', l_orderkey, l_quantity::BIGINT) FROM lineitem").fetchone()[0]
print(f"uploaded {n:,} rows in {time.perf_counter()-t0:.2f} s (one-time)")

# first call pays the one-time sort; every later call reuses it
con.sql("SELECT count(*) FROM gpu_groupby_count_resident('l')").fetchall()

# TPC-H Q18 inner query: HAVING sum(l_quantity) > 300
native_sql = "SELECT count(*) FROM (SELECT l_orderkey, sum(l_quantity::BIGINT) AS q FROM lineitem GROUP BY 1 HAVING q > 300)"
gpu_sql    = "SELECT count(*) FROM gpu_groupby_sum_resident_having('l', '>', 300)"
same = con.sql("""
SELECT (SELECT count(*) FROM ((SELECT key, sum FROM gpu_groupby_sum_resident_having('l','>',300))
                              EXCEPT (SELECT l_orderkey, sum(l_quantity::BIGINT) q FROM lineitem GROUP BY 1 HAVING q > 300))) = 0
   AND (SELECT count(*) FROM ((SELECT l_orderkey, sum(l_quantity::BIGINT) q FROM lineitem GROUP BY 1 HAVING q > 300)
                              EXCEPT (SELECT key, sum FROM gpu_groupby_sum_resident_having('l','>',300)))) = 0
""").fetchone()[0]
show("Q18 inner: GROUP BY + HAVING sum > 300", bench(native_sql), bench(gpu_sql), f"rows equal both ways: {same}")
print(con.sql("SELECT gpu_last_stats()").fetchone()[0])


In [ ]:
# top-10 groups by sum
native_sql = "SELECT l_orderkey, sum(l_quantity::BIGINT) AS q FROM lineitem GROUP BY 1 ORDER BY q DESC LIMIT 10"
gpu_sql    = "SELECT key, sum FROM gpu_groupby_sum_resident_topk('l', 10, 'desc')"
same = [r[1] for r in con.sql(native_sql).fetchall()] == [r[1] for r in con.sql(gpu_sql).fetchall()]
show("top-10 groups by sum (ORDER BY sum DESC LIMIT 10)", bench(native_sql), bench(gpu_sql), f"sums equal: {same}")

# the honest row: returning EVERY group is a smaller win — DuckDB still has to consume them all
native_sql = "SELECT count(*) FROM (SELECT l_orderkey, sum(l_quantity::BIGINT) FROM lineitem GROUP BY 1)"
gpu_sql    = "SELECT count(*) FROM gpu_groupby_sum_resident('l')"
show("all groups returned (millions of rows)", bench(native_sql), bench(gpu_sql), "smaller win by design")


## 4. Fused join + aggregate (v0.5.0)

`SUM(l_extendedprice) FROM lineitem JOIN orders` as one GPU pass against a
device-cached sorted build side — no join output materialised.


In [ ]:
con.sql("SELECT gpu_upload_pair('lp', l_orderkey, (l_extendedprice*100)::BIGINT) FROM lineitem").fetchall()
con.sql("SELECT gpu_upload('o', o_orderkey) FROM orders").fetchall()
native_sql = "SELECT sum((l_extendedprice*100)::BIGINT) FROM lineitem l JOIN orders o ON l.l_orderkey = o.o_orderkey"
gpu_sql    = "SELECT gpu_join_sum_resident('lp.k', 'lp.v', 'o')"
same = con.sql(native_sql).fetchone()[0] == con.sql(gpu_sql).fetchone()[0]
show("JOIN + SUM(BIGINT), 18M x 4.5M", bench(native_sql), bench(gpu_sql), f"bit-equal: {same}")

# EXISTS semi-join: orders that have a late line item
con.sql("SELECT gpu_upload('late', l_orderkey) FROM lineitem WHERE l_receiptdate > l_commitdate").fetchall()
con.sql("SELECT gpu_upload_pair('op', o_orderkey, o_totalprice::DOUBLE) FROM orders").fetchall()
native_sql = "SELECT sum(o_totalprice) FROM orders WHERE EXISTS (SELECT 1 FROM lineitem WHERE l_orderkey = o_orderkey AND l_receiptdate > l_commitdate)"
gpu_sql    = "SELECT gpu_semi_join_sum_resident_f64('op.k', 'op.v', 'late')"
a, b = con.sql(native_sql).fetchone()[0], con.sql(gpu_sql).fetchone()[0]
show("EXISTS semi-join + SUM(DOUBLE)", bench(native_sql), bench(gpu_sql), f"rel-diff {abs(a-b)/abs(a):.1e}")


## 5. Resident-column SUM (v0.4.0) and the honest rows

In [ ]:
con.sql("SELECT gpu_upload('qty', l_quantity::BIGINT) FROM lineitem").fetchall()
native_sql = "SELECT sum(l_quantity::BIGINT) FROM lineitem"
gpu_sql    = "SELECT gpu_sum_resident('qty')"
same = con.sql(native_sql).fetchone()[0] == con.sql(gpu_sql).fetchone()[0]
show("SUM over 18M rows", bench(native_sql), bench(gpu_sql), f"equal: {same}")

# native wins: whole-column MAX answers from zonemap statistics without scanning
native_sql = "SELECT max(l_quantity::BIGINT) FROM lineitem"
gpu_sql    = "SELECT gpu_max_resident('qty')"
show("MAX over a stored column (zonemap)", bench(native_sql), bench(gpu_sql), "native wins — statistics, no scan")


## What you just measured

- Everything above is **statement time against statement time in one process**, best
  of 3 after a warm-up, with every result checked equal to native in the same cell.
- The one-time costs are real: the pair upload and the first sort. They pay off when the
  same data is queried repeatedly — the resident model, not a per-query accelerator.
- The rows where native wins are kept on purpose (whole-column `MAX`, returning every
  group). The full tables for an RTX 4090 and an Apple M4 Max, with raw measurements,
  are in the project's append-only
  [`BENCHMARK.md`](https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/BENCHMARK.md);
  limits and semantics in
  [`KNOWN_ISSUES.md`](https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/KNOWN_ISSUES.md).
- On an Apple Silicon Mac the same SQL runs on the Metal backend — `INSTALL gpudb FROM community`
  there too.

Source and releases: <https://github.com/singhpratech/duckdbgpumetaldbram> — a star helps others find it.
